# Continuous motion (PVT)

The KX2 supports streamed Cartesian trajectories via the Elmo drives' Interpolated Position Mode (IPM). The host samples a target path at the drive's interpolation cadence (8 ms / 125 Hz), pushes (position, velocity) frames into a 16-deep ring buffer ~8 frames ahead of the read pointer, and lets the drive interpolate cubic-Hermite splines between them. The result is smooth, continuous motion — no PPM-style stop-and-restart between segments.

Three APIs use this transport:

| API | Input | Use case |
|---|---|---|
| `arm.move_to_location(..., path='linear')` | one target pose | a single straight-line Cartesian move |
| `arm.backend.move_parametric(fn, duration_s)` | a callable `t → pose` | math-defined paths (figure-8, helix, Lissajous) |
| `arm.backend.move_through_waypoints(poses, speed, accel)` | a list of poses | smooth pass-through any sequence of points |

All three share the same underlying streamer, EMCY handling, and IPM cleanup. See [hello-world](hello-world.ipynb) for the rest of the KX2 API.

## Setup

In [1]:
import math
from pylabrobot.paa.kx2 import KX2, KX2ArmBackend
from pylabrobot.capabilities.arms.standard import CartesianPose
from pylabrobot.resources import Coordinate, Rotation

kx2 = KX2()
await kx2.setup()

uptime library not available, timestamps are relative to boot time and not to Epoch UTC
2026-05-16 21:16:47,364 - pylabrobot.paa.kx2.driver - INFO - canopen: connected, nodes=[1, 2, 3, 4, 6]


## 1. Straight-line move

`path='linear'` on the existing `move_to_location` swaps PPM for the IPM streamer. The gripper tip traces a straight line from current pose to target, sampled at 8 ms intervals along a trapezoidal arc-length profile. `max_gripper_speed` (mm/s) and `max_gripper_acceleration` (mm/s²) are required — the host builds the profile directly from them (no firmware fallback for streamed motion).

End-of-motion drops a few frames of the trajectory tail (~0.3 mm at 50 mm/s); for tighter terminal-position needs, fall through to the joint-mode `move_to_location` (default `path='joint'`).

In [2]:
start = await kx2.arm.request_gripper_pose()
target = Coordinate(x=start.location.x, y=start.location.y, z=start.location.z + 20)
await kx2.arm.move_to_location(
    location=target,
    direction=start.rotation.z,
    backend_params=KX2ArmBackend.CartesianMoveParams(
        max_gripper_speed=50.0, max_gripper_acceleration=200.0, path='linear',
    ),
)

## 2. Parametric trajectories

`move_parametric(path_fn, duration_s)` calls `path_fn(t)` at every IPM sample time (`t ∈ [0, duration_s]` seconds) and streams the resulting poses. The function defines *both* shape and pacing — there's no separate speed/accel cap, so the caller is responsible for keeping derivatives within drive limits.

IK runs on every sample with mandatory joint-limit enforcement; an out-of-range pose anywhere along the trajectory raises `IKError` **before any motion starts**.

### Figure 8 (lemniscate)

In [3]:
start = await kx2.arm.request_gripper_pose()
Rx, Ry, H, T = 25.0, 12.0, 20.0, 10.0  # 50mm wide, 25mm tall lemniscate, 30mm Z arch, 8s

def fig8(t):
    s = t / T
    theta = 2 * math.pi * s
    return CartesianPose(
        location=Coordinate(
            x=start.location.x + Rx * math.sin(theta),
            y=start.location.y + Ry * math.sin(2 * theta) / 2,
            z=start.location.z + H * math.sin(math.pi * s),
        ),
        rotation=start.rotation,
    )

await kx2.arm.backend.move_parametric(fig8, duration_s=T)

### Rising helix

Constant-radius circle in XY, linear ramp in Z. Both endpoints land back at the start XY but the Z is offset.

In [4]:
start = await kx2.arm.request_gripper_pose()
R, dZ, turns, T = 30.0, 40.0, 2.0, 6.0

def helix(t):
    s = t / T
    theta = 2 * math.pi * turns * s
    return CartesianPose(
        location=Coordinate(
            x=start.location.x + R * math.sin(theta),
            y=start.location.y + R * (1 - math.cos(theta)),
            z=start.location.z + dZ * s,
        ),
        rotation=start.rotation,
    )

await kx2.arm.backend.move_parametric(helix, duration_s=T)

### 3D Lissajous

Different per-axis frequencies trace intricate closed (or quasi-closed) curves in space. Tune  for the shape and  for the size; conservative numbers keep elbow/shoulder joint velocity well inside drive limits.

In [5]:
start = await kx2.arm.request_gripper_pose()
Ax, Ay, Az = 12.0, 8.0, 8.0  # small amplitudes keep joint velocity inside drive limits
a, b, c = 2, 1, 1            # frequency ratios
T = 16.0                       # seconds — long enough to keep speeds modest

def lissa(t):
    s = t / T
    return CartesianPose(
        location=Coordinate(
            x=start.location.x + Ax * math.sin(2 * math.pi * a * s),
            y=start.location.y + Ay * math.sin(2 * math.pi * b * s + math.pi / 4),
            z=start.location.z + Az * math.sin(2 * math.pi * c * s),
        ),
        rotation=start.rotation,
    )

await kx2.arm.backend.move_parametric(lissa, duration_s=T)

## 3. Smooth waypoints

`move_through_waypoints(poses, speed, accel)` fits a centripetal Catmull-Rom spline through the list of poses and reparametrizes it with a trapezoidal arc-length profile capped at `speed` (mm/s) and `accel` (mm/s²). The curve is C¹ continuous — there's no stop at intermediate waypoints.

Useful when you have a sequence of named locations (sample sites, plate corners, scan path) and want smooth motion through them.

In [6]:
start = await kx2.arm.request_gripper_pose()

# Square corners + a Z hop at one corner — gripper rounds each corner smoothly
# rather than stopping at each.
def at(dx=0, dy=0, dz=0):
    return CartesianPose(
        location=Coordinate(start.location.x + dx, start.location.y + dy, start.location.z + dz),
        rotation=start.rotation,
    )

waypoints = [
    at(),
    at(dx=20),
    at(dx=20, dy=20),
    at(dx=20, dy=20, dz=15),  # hop up
    at(dx=20, dy=20),         # back down
    at(dy=20),
    at(),
]

await kx2.arm.backend.move_through_waypoints(waypoints, speed=25.0, accel=80.0)

### Zig-zag scan

Useful pattern for sweeping over a rectangular region (e.g. raster scanning a plate). The Catmull-Rom spline rounds each corner so the gripper never has to come to a stop.

In [7]:
start = await kx2.arm.request_gripper_pose()
rows, row_dy, sweep_dx = 4, 10.0, 50.0

waypoints = []
for r in range(rows):
    y = r * row_dy
    x_left, x_right = (0.0, sweep_dx) if r % 2 == 0 else (sweep_dx, 0.0)
    waypoints.append(CartesianPose(
        location=Coordinate(start.location.x + x_left, start.location.y + y, start.location.z),
        rotation=start.rotation,
    ))
    waypoints.append(CartesianPose(
        location=Coordinate(start.location.x + x_right, start.location.y + y, start.location.z),
        rotation=start.rotation,
    ))

await kx2.arm.backend.move_through_waypoints(waypoints, speed=30.0, accel=100.0)

## Safety notes

- **Joint limits are enforced by IK**: any pose whose IK solution falls outside the per-axis `[min_travel, max_travel]` raises `IKError` before any frame leaves the host. Catches workspace-violating trajectories at sample time, not at the drive.
- **Drive velocity / acceleration limits are NOT enforced**. `move_parametric` lets the caller define the velocity profile, and `move_through_waypoints` only caps Cartesian speed/accel — neither checks per-axis encoder rates. A swirly trajectory near a kinematic singularity can demand joint speeds way above what the motors can produce; the drives will either lag or fault. Stay inside the workspace and use moderate speed/accel.
- **Cancel/halt during streaming** drops ip-enable and lets the drive consume already-buffered frames before stopping. Coast can run up to ~64 ms past the cancel. For true zero-coast stop, use `await kx2.arm.backend.halt()` (MO=0 on every motion axis).

## Teardown

In [8]:
await kx2.stop()